# 06 — Causal Pruning: Fisher Information-Guided Task Isolation

## Motivation

PackNet protects weights by *magnitude* — the larger the weight, the more likely it is to be kept.
But magnitude is a purely *observational* criterion.  A large weight may be largely redundant for
a task (the network could learn the same function without it), while a small weight may be
causally critical (removing it would collapse task performance).

## Causal Importance via Fisher Information

The **diagonal empirical Fisher score** measures how sensitive the loss is to each weight:

$$F_i = \mathbb{E}_{(x,y)\sim\mathcal{D}_t}\!\left[\left(\frac{\partial \mathcal{L}}{\partial \theta_i}\right)^2\right]$$

This is an *interventional* measure: $F_i$ is large if and only if perturbing $\theta_i$ changes
the task loss.  Protecting high-$F_i$ weights is causally justified — these weights causally
determine task performance.

## Algorithm

1. Train task $t$ with gradient masking (only free weights updated).
2. Compute $F_i$ for all free weights via squared gradient accumulation.
3. Sort free weights by $F_i$ (descending).
4. Protect the top `keep_ratio` fraction → assigned to task $t$, frozen hereafter.
5. The remaining free weights are available for task $t+1$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from src.preprocess import get_split_cifar100
from src.models.base_model import get_model
from src.models.causal_pruning import CausalPruner
from src.train import run_continual
from src.evaluate import backward_transfer, forward_transfer, average_accuracy, print_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Visualising Fisher Scores

Before running the full continual learning loop, we visualise what the Fisher scores look like
after training a single task.

In [ ]:
# Train a small model on task 0 only to inspect Fisher scores
import torch.nn as nn, torch.optim as optim

tasks = get_split_cifar100(n_tasks=20, batch_size=128, seed=42, data_root='../data')
model = get_model('cifar100', n_tasks=20, classes_per_task=5).to(device)

# Quick training: 3 epochs on task 0
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(3):
    for x, y in tasks[0].train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, 0), y)
        loss.backward()
        optimizer.step()
print('Task-0 training done.')

# Compute Fisher scores
cp = CausalPruner(keep_ratio=0.5, n_samples=2048)
cp.register(model)
fisher = cp.get_causal_scores(model, tasks[0].train_loader, device, task_id=0)
print(f'Fisher computed for {len(fisher)} parameter tensors.')

In [ ]:
# Plot Fisher score distribution for conv layers vs FC layers
conv_scores, fc_scores = [], []

for name, f in fisher.items():
    flat = f.cpu().numpy().ravel()
    if 'conv' in name or 'layer' in name:
        conv_scores.append(flat)
    elif 'head' in name or 'linear' in name.lower():
        fc_scores.append(flat)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if conv_scores:
    all_conv = np.concatenate(conv_scores)
    axes[0].hist(np.log10(all_conv + 1e-12), bins=60, color='#3498db', alpha=0.8, edgecolor='white')
    axes[0].set_xlabel('log₁₀(Fisher score)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Conv layer Fisher scores\n(Task 0, 3 epochs)', fontweight='bold')
    axes[0].grid(True, alpha=0.3)

if fc_scores:
    all_fc = np.concatenate(fc_scores)
    axes[1].hist(np.log10(all_fc + 1e-12), bins=60, color='#9b59b6', alpha=0.8, edgecolor='white')
    axes[1].set_xlabel('log₁₀(Fisher score)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Task-head Fisher scores\n(Task 0, 3 epochs)', fontweight='bold')
    axes[1].grid(True, alpha=0.3)

plt.suptitle('Empirical Fisher Score Distribution (Causal Importance)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/improved/fisher_score_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Magnitude vs Fisher score scatter for a single layer
layer_name = [n for n in fisher if 'layer1' in n and 'conv' in n][0]
f_vals = fisher[layer_name].cpu().numpy().ravel()
w_vals = dict(model.named_parameters())[layer_name].data.cpu().numpy().ravel()

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(np.abs(w_vals), f_vals, alpha=0.3, s=8, c=np.log10(f_vals + 1e-12),
               cmap='plasma')
plt.colorbar(sc, ax=ax, label='log₁₀(Fisher)')
ax.set_xlabel('|Weight| (magnitude)')
ax.set_ylabel('Fisher score (causal importance)')
ax.set_title(f'Magnitude vs Causal Importance\n{layer_name}', fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Annotate: high Fisher but low magnitude (these are the weights PackNet misses)
interesting = (f_vals > np.percentile(f_vals, 90)) & (np.abs(w_vals) < np.percentile(np.abs(w_vals), 30))
ax.scatter(np.abs(w_vals[interesting]), f_vals[interesting],
           s=30, color='red', zorder=5, label='High Fisher, low magnitude')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../results/improved/magnitude_vs_fisher.png', bbox_inches='tight')
plt.show()

corr = np.corrcoef(np.abs(w_vals), f_vals)[0, 1]
print(f'Pearson correlation |weight| vs Fisher: {corr:.4f}')
print(f'(low correlation supports using Fisher rather than magnitude for selection)')

## 2. Full Continual Learning Run

In [ ]:
R_cp = run_continual(
    method='causal',
    dataset='cifar100',
    n_tasks=20,
    epochs_per_task=10,
    lr=0.1,
    batch_size=128,
    device=device,
    keep_ratio=0.5,    # protect top 50% of free weights by Fisher score
    n_fisher=1024,
    seed=42,
    save_dir='../results/improved',
    verbose=True,
)

## 3. Keep-Ratio Ablation

In [ ]:
keep_ratios = [0.3, 0.4, 0.5, 0.6, 0.7]
cp_avgs, cp_bwts = [], []

for kr in keep_ratios:
    print(f'  keep_ratio={kr} ...', end='')
    R = run_continual(
        method='causal', dataset='cifar100', n_tasks=5,
        epochs_per_task=5, lr=0.1, batch_size=128,
        device=device, keep_ratio=kr, n_fisher=512, seed=42,
        save_dir='../results/improved', verbose=False,
    )
    cp_avgs.append(average_accuracy(R) * 100)
    cp_bwts.append(backward_transfer(R) * 100)
    print(f' avg_acc={cp_avgs[-1]:.1f}%  bwt={cp_bwts[-1]:.1f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(keep_ratios, cp_avgs, marker='o', color='#9b59b6', linewidth=2, label='CausalPruner')
ax1.set_xlabel('Keep ratio'); ax1.set_ylabel('Avg Accuracy (%)')
ax1.set_title('CausalPruner: Accuracy vs Keep Ratio'); ax1.grid(True, alpha=0.3)

ax2.plot(keep_ratios, cp_bwts, marker='s', color='#e74c3c', linewidth=2)
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Keep ratio'); ax2.set_ylabel('BWT (%)')
ax2.set_title('CausalPruner: BWT vs Keep Ratio'); ax2.grid(True, alpha=0.3)

plt.suptitle('CausalPruner Keep-Ratio Ablation (5 tasks)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/improved/causal_keep_ablation.png', bbox_inches='tight')
plt.show()

## 4. Metrics

In [ ]:
print_metrics(R_cp, method='CausalPruner (keep_ratio=0.5)')

## 5. CausalPruner vs PackNet: Head-to-Head

In [ ]:
# Load PackNet result if it exists
import os, numpy as np
pn_path = '../results/improved/packnet_cifar100_acc_matrix.npy'
cp_path = '../results/improved/causal_cifar100_acc_matrix.npy'

if os.path.exists(pn_path) and os.path.exists(cp_path):
    R_pn2 = np.load(pn_path)
    R_cp2 = np.load(cp_path)
    T = R_cp2.shape[0]
    
    # Forgetting curves for both
    fig, ax = plt.subplots(figsize=(9, 5))
    pn_curve  = [R_pn2[i, 0] * 100 for i in range(T)]
    cp_curve  = [R_cp2[i, 0] * 100 for i in range(T)]
    ax.plot(range(T), pn_curve, marker='s', color='#2ecc71', linewidth=2, label='PackNet')
    ax.plot(range(T), cp_curve, marker='o', color='#9b59b6', linewidth=2, label='CausalPruner')
    ax.set_xlabel('Tasks trained'); ax.set_ylabel('Accuracy on Task 0 (%)')
    ax.set_title('PackNet vs CausalPruner: Task-0 Retention', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-5, 105)
    plt.tight_layout()
    plt.savefig('../results/improved/packnet_vs_causal.png', bbox_inches='tight')
    plt.show()
else:
    print('Run notebook 05 first to generate PackNet results.')